In [1]:
import json

# Path dataset di Google Colab
DATASET_PATH = "/content/ground_truth.json"

# Read dataset
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Total data: {len(dataset)}")
print(dataset[0])

Total data: 18
{'full_sentence': 'paman sedang repair mesin mobil merah milik tetangga sebelah rumah', 'words': [{'word': 'paman', 'language': 'indonesian'}, {'word': 'sedang', 'language': 'indonesian'}, {'word': 'repair', 'language': 'english'}, {'word': 'mesin', 'language': 'indonesian'}, {'word': 'mobil', 'language': 'indonesian'}, {'word': 'merah', 'language': 'indonesian'}, {'word': 'milik', 'language': 'indonesian'}, {'word': 'tetangga', 'language': 'indonesian'}, {'word': 'sebelah', 'language': 'indonesian'}, {'word': 'rumah', 'language': 'indonesian'}], 'en_percentage': '10%', 'id_percentage': '90%'}


In [2]:
print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

{
  "full_sentence": "paman sedang repair mesin mobil merah milik tetangga sebelah rumah",
  "words": [
    {
      "word": "paman",
      "language": "indonesian"
    },
    {
      "word": "sedang",
      "language": "indonesian"
    },
    {
      "word": "repair",
      "language": "english"
    },
    {
      "word": "mesin",
      "language": "indonesian"
    },
    {
      "word": "mobil",
      "language": "indonesian"
    },
    {
      "word": "merah",
      "language": "indonesian"
    },
    {
      "word": "milik",
      "language": "indonesian"
    },
    {
      "word": "tetangga",
      "language": "indonesian"
    },
    {
      "word": "sebelah",
      "language": "indonesian"
    },
    {
      "word": "rumah",
      "language": "indonesian"
    }
  ],
  "en_percentage": "10%",
  "id_percentage": "90%"
}


In [3]:
import pandas as pd

PARQUET_PATH = "/content/sessions_lang_transcript.parquet"

df = pd.read_parquet(PARQUET_PATH)

# Tampilkan 1 baris pertama secara penuh, tanpa truncation
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
    "display.max_rows", None
):
    display(df.head(1))

gamesession_id  user_id  game_name model_type           created_at  \
0       140597768   830382  Minecraft      gen10  2026-08-04 13:05:17   

  lang_detected  lang_probability  \
0            en             0.999   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [4]:
import requests
import json
import re
import pandas as pd
from google.colab import userdata

# ============================================================
# CONFIG
# ============================================================

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

MODEL = "tencent/hy-mt2-30b-a3b"
API_URL = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}


# ============================================================
# LLM LANGUAGE DETECTION
# ============================================================

def detect_language_proportion(sentence):
    prompt = f"""
Analyze the language composition of the following sentence.

The sentence may contain Indonesian and English words.

Your tasks:
1. Identify the language of EVERY word.
2. Classify each word as exactly one of:
   - indonesian
   - english
3. Calculate the percentage of Indonesian and English words.
4. Percentages MUST be based on the number of words.
5. Indonesian percentage + English percentage MUST equal 100.
6. Return ONLY valid JSON. Do not return markdown or explanation.

Sentence:
{sentence}

Return exactly this structure:

{{
  "words": [
    {{
      "word": "example",
      "language": "english"
    }}
  ],
  "en_percentage": 0,
  "id_percentage": 0
}}
"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "reasoning": {
            "enabled": True
        }
    }

    response = requests.post(
        API_URL,
        headers=headers,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    response_json = response.json()
    content = response_json["choices"][0]["message"]["content"]

    # Bersihkan jika model masih mengembalikan ```json ... ```
    content = re.sub(r"^```json\s*", "", content.strip())
    content = re.sub(r"^```\s*", "", content)
    content = re.sub(r"\s*```$", "", content)

    result = json.loads(content)

    return result


# ============================================================
# RUN EXPERIMENT
# ============================================================

results = []

for i, item in enumerate(dataset):

    sentence = item["full_sentence"]

    print(f"[{i + 1}/{len(dataset)}] {sentence}")

    try:
        prediction = detect_language_proportion(sentence)

        # Ground truth
        gt_en = int(item["en_percentage"].replace("%", ""))
        gt_id = int(item["id_percentage"].replace("%", ""))

        # Prediction
        pred_en = float(prediction["en_percentage"])
        pred_id = float(prediction["id_percentage"])

        # Absolute percentage error
        en_error = abs(pred_en - gt_en)
        id_error = abs(pred_id - gt_id)

        # ====================================================
        # WORD-LEVEL ACCURACY
        # ====================================================

        gt_words = item["words"]
        pred_words = prediction["words"]

        correct_words = 0
        total_words = len(gt_words)

        for gt_word, pred_word in zip(gt_words, pred_words):

            if (
                gt_word["word"].lower() == pred_word["word"].lower()
                and
                gt_word["language"].lower() == pred_word["language"].lower()
            ):
                correct_words += 1

        word_accuracy = (
            correct_words / total_words * 100
            if total_words > 0
            else 0
        )

        results.append({
            "sentence": sentence,

            "gt_en_percentage": gt_en,
            "pred_en_percentage": pred_en,
            "en_error": en_error,

            "gt_id_percentage": gt_id,
            "pred_id_percentage": pred_id,
            "id_error": id_error,

            "word_accuracy": word_accuracy,

            "prediction_words": prediction["words"]
        })

    except Exception as e:
        print("ERROR:", e)

        results.append({
            "sentence": sentence,
            "error": str(e)
        })


# ============================================================
# RESULT DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

display(results_df)

[1/18] paman sedang repair mesin mobil merah milik tetangga sebelah rumah
[2/18] saya beli outfit warna biru muda untuk photoshoot besok pagi
[3/18] tolong share jadwal meeting ini ke grup tim secara online
[4/18] saya need to check berkas proyek ini before kita kirim
[5/18] please update jadwal rapat ini because supervisor saya delay datang
[6/18] kamu can learn new skill ini directly from pelatih terbaik
[7/18] they will fix jalan rusak this morning dekat my house
[8/18] we need to edit camera setting ini before shooting besok
[9/18] you need to adjust the pencahayaan for this photo session
[10/18] kakak sedang design undangan pesta ulang tahun untuk adik kecil
[11/18] ibu ingin beli laptop baru untuk kerja online setiap hari
[12/18] saya download file penting sebelum rapat online dimulai nanti sore
[13/18] we need test aplikasi baru sebelum launch besok pagi ini
[14/18] please confirm jadwal kita because client saya complain kemarin sore
[15/18] kamu should always practice speaking 

,sentence,gt_en_percentage,pred_en_percentage,en_error,gt_id_percentage,pred_id_percentage,id_error,word_accuracy,prediction_words
0,paman sedang repair mesin mobil merah milik te...,10,10.0,0.0,90,90.0,0.0,100.0,"[{'word': 'paman', 'language': 'indonesian'}, ..."
1,saya beli outfit warna biru muda untuk photosh...,20,20.0,0.0,80,80.0,0.0,100.0,"[{'word': 'saya', 'language': 'indonesian'}, {..."
2,tolong share jadwal meeting ini ke grup tim se...,30,30.0,0.0,70,70.0,0.0,100.0,"[{'word': 'tolong', 'language': 'indonesian'},..."
3,saya need to check berkas proyek ini before ki...,40,50.0,10.0,60,50.0,10.0,100.0,"[{'word': 'saya', 'language': 'indonesian'}, {..."
4,please update jadwal rapat ini because supervi...,50,60.0,10.0,50,40.0,10.0,100.0,"[{'word': 'please', 'language': 'english'}, {'..."
5,kamu can learn new skill ini directly from pel...,60,70.0,10.0,40,30.0,10.0,100.0,"[{'word': 'kamu', 'language': 'indonesian'}, {..."
6,they will fix jalan rusak this morning dekat m...,70,70.0,0.0,30,30.0,0.0,100.0,"[{'word': 'they', 'language': 'english'}, {'wo..."
7,we need to edit camera setting ini before shoo...,80,70.0,10.0,20,30.0,10.0,100.0,"[{'word': 'we', 'language': 'english'}, {'word..."
8,you need to adjust the pencahayaan for this ph...,90,90.0,0.0,10,10.0,0.0,100.0,"[{'word': 'you', 'language': 'english'}, {'wor..."
9,kakak sedang design undangan pesta ulang tahun...,10,10.0,0.0,90,90.0,0.0,100.0,"[{'word': 'kakak', 'language': 'indonesian'}, ..."


In [5]:
# Ambil hanya request yang berhasil
valid_results = results_df[
    results_df["en_error"].notna()
].copy()

# Mean Absolute Error dalam percentage point
en_mae = valid_results["en_error"].mean()
id_mae = valid_results["id_error"].mean()

# Rata-rata akurasi klasifikasi tiap kata
avg_word_accuracy = valid_results["word_accuracy"].mean()

# Exact percentage match
exact_match = (
    (valid_results["en_error"] == 0) &
    (valid_results["id_error"] == 0)
).mean() * 100

# Toleransi +/- 10 percentage point
within_10 = (
    (valid_results["en_error"] <= 10) &
    (valid_results["id_error"] <= 10)
).mean() * 100


print("=" * 50)
print("EXPERIMENT RESULT")
print("=" * 50)

print(f"Total samples          : {len(valid_results)}")
print(f"English MAE            : {en_mae:.2f} percentage points")
print(f"Indonesian MAE         : {id_mae:.2f} percentage points")
print(f"Word-level accuracy    : {avg_word_accuracy:.2f}%")
print(f"Exact percentage match : {exact_match:.2f}%")
print(f"Within ±10 points      : {within_10:.2f}%")

EXPERIMENT RESULT
Total samples          : 18
English MAE            : 5.00 percentage points
Indonesian MAE         : 5.00 percentage points
Word-level accuracy    : 100.00%
Exact percentage match : 50.00%
Within ±10 points      : 100.00%


In [6]:
comparison = valid_results[
    [
        "sentence",
        "gt_en_percentage",
        "pred_en_percentage",
        "en_error",
        "gt_id_percentage",
        "pred_id_percentage",
        "id_error",
        "word_accuracy"
    ]
].sort_values(
    "en_error",
    ascending=False
)

display(comparison)

,sentence,gt_en_percentage,pred_en_percentage,en_error,gt_id_percentage,pred_id_percentage,id_error,word_accuracy
4,please update jadwal rapat ini because supervi...,50,60.0,10.0,50,40.0,10.0,100.0
3,saya need to check berkas proyek ini before ki...,40,50.0,10.0,60,50.0,10.0,100.0
16,we should finish this important project before...,80,90.0,10.0,20,10.0,10.0,100.0
13,please confirm jadwal kita because client saya...,50,60.0,10.0,50,40.0,10.0,100.0
5,kamu can learn new skill ini directly from pel...,60,70.0,10.0,40,30.0,10.0,100.0
7,we need to edit camera setting ini before shoo...,80,70.0,10.0,20,30.0,10.0,100.0
14,kamu should always practice speaking english d...,60,70.0,10.0,40,30.0,10.0,100.0
11,saya download file penting sebelum rapat onlin...,30,40.0,10.0,70,60.0,10.0,100.0
12,we need test aplikasi baru sebelum launch beso...,40,50.0,10.0,60,50.0,10.0,100.0
0,paman sedang repair mesin mobil merah milik te...,10,10.0,0.0,90,90.0,0.0,100.0


In [7]:
import json
import re
import requests
import pandas as pd
from google.colab import userdata


# ============================================================
# CONFIG
# ============================================================

DATASET_PATH = "/content/ground_truth.json"

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

MODEL = "tencent/hy-mt2-30b-a3b"
API_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}


# ============================================================
# READ GROUND TRUTH
# ============================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Total samples: {len(dataset)}")

Total samples: 18


In [8]:
def predict_language_words(sentence):
    """
    IMPORTANT:
    Model hanya menerima `sentence`.
    Ground-truth labels/persentase TIDAK dikirim ke model.
    """

    prompt = f"""
Identify the language of EVERY word in the sentence below.

For each word, classify it as exactly one of:
- indonesian
- english

Rules:
1. Preserve every word exactly as written.
2. Preserve the original word order.
3. Do not skip any word.
4. Do not add any word.
5. Return ONLY valid JSON.
6. Do NOT calculate percentages.

Sentence:
{sentence}

Return exactly this JSON structure:

{{
  "words": [
    {{
      "word": "example",
      "language": "english"
    }}
  ]
}}
"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "reasoning": {
            "enabled": True
        }
    }

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    response_json = response.json()

    content = response_json["choices"][0]["message"]["content"]

    # Bersihkan kemungkinan markdown code fence
    content = content.strip()
    content = re.sub(r"^```json\s*", "", content)
    content = re.sub(r"^```\s*", "", content)
    content = re.sub(r"\s*```$", "", content)

    return json.loads(content)

In [9]:
def calculate_model_stats(predicted_words):
    total = len(predicted_words)

    en_count = sum(
        1 for item in predicted_words
        if item["language"].lower() == "english"
    )

    id_count = sum(
        1 for item in predicted_words
        if item["language"].lower() == "indonesian"
    )

    en_percentage = (en_count / total * 100) if total else 0
    id_percentage = (id_count / total * 100) if total else 0

    return {
        "total_words": total,
        "en_count": en_count,
        "id_count": id_count,
        "en_percentage": en_percentage,
        "id_percentage": id_percentage,
    }

In [10]:
results = []

for i, item in enumerate(dataset):

    # ========================================================
    # MODEL SIDE
    # HANYA full_sentence yang diambil
    # ========================================================

    sentence = item["full_sentence"]

    print(f"[{i + 1}/{len(dataset)}] {sentence}")

    try:
        prediction = predict_language_words(sentence)

        model_stats = calculate_model_stats(
            prediction["words"]
        )


        # ====================================================
        # EVALUATOR SIDE
        # Ground truth BARU digunakan setelah prediksi model
        # ====================================================

        gt_words = item["words"]

        gt_en_count = sum(
            1 for word in gt_words
            if word["language"].lower() == "english"
        )

        gt_id_count = sum(
            1 for word in gt_words
            if word["language"].lower() == "indonesian"
        )

        gt_total = len(gt_words)

        # Ambil persentase yang memang tersimpan di JSON
        gt_en_percentage = float(
            item["en_percentage"].replace("%", "")
        )

        gt_id_percentage = float(
            item["id_percentage"].replace("%", "")
        )


        # ====================================================
        # COMPARE
        # ====================================================

        en_count_error = abs(
            model_stats["en_count"] - gt_en_count
        )

        id_count_error = abs(
            model_stats["id_count"] - gt_id_count
        )

        en_percentage_error = abs(
            model_stats["en_percentage"] - gt_en_percentage
        )

        id_percentage_error = abs(
            model_stats["id_percentage"] - gt_id_percentage
        )


        # ====================================================
        # WORD-LEVEL COMPARISON
        # ====================================================

        correct_labels = 0

        if len(prediction["words"]) == len(gt_words):

            for gt_word, pred_word in zip(
                gt_words,
                prediction["words"]
            ):

                same_word = (
                    gt_word["word"].lower()
                    == pred_word["word"].lower()
                )

                same_language = (
                    gt_word["language"].lower()
                    == pred_word["language"].lower()
                )

                if same_word and same_language:
                    correct_labels += 1

        word_accuracy = (
            correct_labels / gt_total * 100
            if gt_total > 0
            else 0
        )


        results.append({

            "sentence": sentence,

            # Ground truth
            "gt_total_words": gt_total,
            "gt_en_count": gt_en_count,
            "gt_id_count": gt_id_count,
            "gt_en_percentage": gt_en_percentage,
            "gt_id_percentage": gt_id_percentage,

            # Model
            "model_total_words": model_stats["total_words"],
            "model_en_count": model_stats["en_count"],
            "model_id_count": model_stats["id_count"],
            "model_en_percentage": model_stats["en_percentage"],
            "model_id_percentage": model_stats["id_percentage"],

            # Error
            "en_count_error": en_count_error,
            "id_count_error": id_count_error,
            "en_percentage_error": en_percentage_error,
            "id_percentage_error": id_percentage_error,

            # Word level
            "word_accuracy": word_accuracy,

            # Untuk inspection
            "model_words": prediction["words"],
        })


    except Exception as e:

        print("ERROR:", e)

        results.append({
            "sentence": sentence,
            "error": str(e)
        })


results_df = pd.DataFrame(results)

[1/18] paman sedang repair mesin mobil merah milik tetangga sebelah rumah
[2/18] saya beli outfit warna biru muda untuk photoshoot besok pagi
[3/18] tolong share jadwal meeting ini ke grup tim secara online
[4/18] saya need to check berkas proyek ini before kita kirim
[5/18] please update jadwal rapat ini because supervisor saya delay datang
[6/18] kamu can learn new skill ini directly from pelatih terbaik
[7/18] they will fix jalan rusak this morning dekat my house
[8/18] we need to edit camera setting ini before shooting besok
[9/18] you need to adjust the pencahayaan for this photo session
[10/18] kakak sedang design undangan pesta ulang tahun untuk adik kecil
[11/18] ibu ingin beli laptop baru untuk kerja online setiap hari
[12/18] saya download file penting sebelum rapat online dimulai nanti sore
[13/18] we need test aplikasi baru sebelum launch besok pagi ini
[14/18] please confirm jadwal kita because client saya complain kemarin sore
[15/18] kamu should always practice speaking 

In [11]:
comparison_columns = [
    "sentence",

    "gt_en_count",
    "model_en_count",

    "gt_id_count",
    "model_id_count",

    "gt_en_percentage",
    "model_en_percentage",

    "gt_id_percentage",
    "model_id_percentage",

    "en_percentage_error",
    "id_percentage_error",

    "word_accuracy"
]

display(
    results_df[comparison_columns]
)

,sentence,gt_en_count,model_en_count,gt_id_count,model_id_count,gt_en_percentage,model_en_percentage,gt_id_percentage,model_id_percentage,en_percentage_error,id_percentage_error,word_accuracy
0,paman sedang repair mesin mobil merah milik te...,1,1,9,9,10.0,10.0,90.0,90.0,0.0,0.0,100.0
1,saya beli outfit warna biru muda untuk photosh...,2,2,8,8,20.0,20.0,80.0,80.0,0.0,0.0,100.0
2,tolong share jadwal meeting ini ke grup tim se...,3,2,7,8,30.0,20.0,70.0,80.0,10.0,10.0,90.0
3,saya need to check berkas proyek ini before ki...,4,4,6,6,40.0,40.0,60.0,60.0,0.0,0.0,100.0
4,please update jadwal rapat ini because supervi...,5,5,5,5,50.0,50.0,50.0,50.0,0.0,0.0,100.0
5,kamu can learn new skill ini directly from pel...,6,6,4,4,60.0,60.0,40.0,40.0,0.0,0.0,100.0
6,they will fix jalan rusak this morning dekat m...,7,7,3,3,70.0,70.0,30.0,30.0,0.0,0.0,100.0
7,we need to edit camera setting ini before shoo...,8,8,2,2,80.0,80.0,20.0,20.0,0.0,0.0,100.0
8,you need to adjust the pencahayaan for this ph...,9,9,1,1,90.0,90.0,10.0,10.0,0.0,0.0,100.0
9,kakak sedang design undangan pesta ulang tahun...,1,1,9,9,10.0,10.0,90.0,90.0,0.0,0.0,100.0


In [12]:
valid_results = results_df[
    results_df["en_percentage_error"].notna()
].copy()


en_mae = valid_results[
    "en_percentage_error"
].mean()

id_mae = valid_results[
    "id_percentage_error"
].mean()

avg_word_accuracy = valid_results[
    "word_accuracy"
].mean()


exact_count_match = (
    (valid_results["gt_en_count"] == valid_results["model_en_count"])
    &
    (valid_results["gt_id_count"] == valid_results["model_id_count"])
).mean() * 100


exact_percentage_match = (
    (valid_results["en_percentage_error"] < 1e-9)
    &
    (valid_results["id_percentage_error"] < 1e-9)
).mean() * 100


print("=" * 60)
print("BLIND LANGUAGE DETECTION EVALUATION")
print("=" * 60)

print(f"Total samples              : {len(valid_results)}")
print(f"English percentage MAE     : {en_mae:.2f} points")
print(f"Indonesian percentage MAE  : {id_mae:.2f} points")
print(f"Average word accuracy      : {avg_word_accuracy:.2f}%")
print(f"Exact word-count match     : {exact_count_match:.2f}%")
print(f"Exact percentage match     : {exact_percentage_match:.2f}%")

BLIND LANGUAGE DETECTION EVALUATION
Total samples              : 18
English percentage MAE     : 1.67 points
Indonesian percentage MAE  : 1.67 points
Average word accuracy      : 98.33%
Exact word-count match     : 88.89%
Exact percentage match     : 88.89%


In [13]:
errors = valid_results[
    (valid_results["en_count_error"] > 0)
    |
    (valid_results["id_count_error"] > 0)
].copy()

errors = errors.sort_values(
    "en_percentage_error",
    ascending=False
)

display(
    errors[
        [
            "sentence",
            "gt_en_count",
            "model_en_count",
            "gt_id_count",
            "model_id_count",
            "gt_en_percentage",
            "model_en_percentage",
            "word_accuracy",
            "model_words"
        ]
    ]
)

,sentence,gt_en_count,model_en_count,gt_id_count,model_id_count,gt_en_percentage,model_en_percentage,word_accuracy,model_words
10,ibu ingin beli laptop baru untuk kerja online ...,2,0,8,10,20.0,0.0,80.0,"[{'word': 'ibu', 'language': 'indonesian'}, {'..."
2,tolong share jadwal meeting ini ke grup tim se...,3,2,7,8,30.0,20.0,90.0,"[{'word': 'tolong', 'language': 'indonesian'},..."


In [14]:
sentence = item["full_sentence"]

prediction = predict_language_words(sentence)

In [15]:
import pandas as pd
from IPython.display import display, Markdown

for i, item in enumerate(dataset, start=1):

    # ============================================================
    # INPUT UNTUK LLM
    # HANYA KALIMAT ASLI
    # ============================================================

    sentence = item["full_sentence"]

    # Model TIDAK melihat:
    # - item["words"]
    # - item["en_percentage"]
    # - item["id_percentage"]

    prediction = predict_language_words(sentence)

    predicted_words = prediction["words"]


    # ============================================================
    # BARU SETELAH MODEL SELESAI:
    # AMBIL GROUND TRUTH UNTUK EVALUASI / DISPLAY
    # ============================================================

    ground_truth_words = item["words"]


    # ============================================================
    # TABEL 1
    # DATA ASLI + LABEL ASLI
    # ============================================================

    table_1 = pd.DataFrame([
        {
            "word": x["word"],
            "label": x["language"]
        }
        for x in ground_truth_words
    ])


    # ============================================================
    # TABEL 2
    # DATA ASLI + PERSENTASE ASLI
    # ============================================================

    table_2 = pd.DataFrame([
        {
            "eng": item["en_percentage"],
            "id": item["id_percentage"]
        }
    ])


    # ============================================================
    # TABEL 3
    # HASIL LLM + LABEL HASIL TEBAKAN LLM
    # ============================================================

    table_3 = pd.DataFrame([
        {
            "word": x["word"],
            "label": x["language"]
        }
        for x in predicted_words
    ])


    # ============================================================
    # HITUNG PERSENTASE HASIL LLM
    # PYTHON YANG MENGHITUNG, BUKAN LLM
    # ============================================================

    total_words = len(predicted_words)

    model_eng_count = sum(
        x["language"].lower() == "english"
        for x in predicted_words
    )

    model_id_count = sum(
        x["language"].lower() == "indonesian"
        for x in predicted_words
    )

    model_eng_percentage = (
        model_eng_count / total_words * 100
        if total_words > 0
        else 0
    )

    model_id_percentage = (
        model_id_count / total_words * 100
        if total_words > 0
        else 0
    )


    # ============================================================
    # TABEL 4
    # PERSENTASE HASIL LLM
    # ============================================================

    table_4 = pd.DataFrame([
        {
            "eng": f"{model_eng_percentage:.0f}%",
            "id": f"{model_id_percentage:.0f}%"
        }
    ])


    # ============================================================
    # OUTPUT
    # ============================================================

    display(
        Markdown(
            f"## Sample {i}\n\n"
            f"**Sentence:** `{sentence}`"
        )
    )

    display(Markdown("### Tabel 1 — Ground Truth: Word + Label"))
    display(table_1)

    display(Markdown("### Tabel 2 — Ground Truth: Percentage"))
    display(table_2)

    display(Markdown("### Tabel 3 — LLM Prediction: Word + Label"))
    display(table_3)

    display(Markdown("### Tabel 4 — LLM Prediction: Percentage"))
    display(table_4)

## Sample 1

**Sentence:** `paman sedang repair mesin mobil merah milik tetangga sebelah rumah`

### Tabel 1 — Ground Truth: Word + Label

,word,label
0,paman,indonesian
1,sedang,indonesian
2,repair,english
3,mesin,indonesian
4,mobil,indonesian
5,merah,indonesian
6,milik,indonesian
7,tetangga,indonesian
8,sebelah,indonesian
9,rumah,indonesian


### Tabel 2 — Ground Truth: Percentage

,eng,id
0,10%,90%


...

,eng,id
0,90%,10%


### Tabel 3 — LLM Prediction: Word + Label

,word,label
0,you,english
1,should,english
2,always,english
3,bring,english
4,charger,english
5,for,english
6,this,english
7,important,english
8,meeting,english
9,nanti,indonesian


### Tabel 4 — LLM Prediction: Percentage

,eng,id
0,90%,10%


In [16]:
import pandas as pd
import requests
import json
import re
from google.colab import userdata
from IPython.display import display


# ============================================================
# CONFIG
# ============================================================

PARQUET_PATH = "/content/sessions_lang_transcript.parquet"

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

MODEL = "tencent/hy-mt2-30b-a3b"
API_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}


# ============================================================
# READ PARQUET
# ============================================================

df = pd.read_parquet(PARQUET_PATH)


# ============================================================
# AMBIL 1 SESSION
# ============================================================

session_row = df.iloc[0]

session_id = session_row["gamesession_id"]
segments = session_row["transcript_segments"]


# hanya ambil 5 segment pertama
segments = segments[:5]

print("Session ID:", session_id)
print("Total segment yang diproses:", len(segments))

Session ID: 140597768
Total segment yang diproses: 5


In [17]:
def detect_segment_languages(segment_text):

    prompt = f"""
Analyze every word in the transcript segment below.

For every word, classify its language.

Use short ISO-like language codes when possible, for example:
- en = English
- id = Indonesian
- de = German
- es = Spanish
- fr = French
- ja = Japanese
- ko = Korean
- zh = Chinese

Rules:
1. Preserve every word exactly as written.
2. Preserve the original word order.
3. Do not skip words.
4. Do not add words.
5. Detect the actual language of each word.
6. Return ONLY valid JSON.
7. Do NOT calculate percentages.

Transcript segment:
{segment_text}

Return exactly:

{{
  "words": [
    {{
      "word": "example",
      "language": "en"
    }}
  ]
}}
"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "reasoning": {
            "enabled": True
        }
    }

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    response_json = response.json()
    content = response_json["choices"][0]["message"]["content"]

    content = content.strip()
    content = re.sub(r"^```json\s*", "", content)
    content = re.sub(r"^```\s*", "", content)
    content = re.sub(r"\s*```$", "", content)

    return json.loads(content)

In [18]:
def calculate_language_percentages(predicted_words):

    total_words = len(predicted_words)

    language_counts = {}

    for item in predicted_words:
        lang = item["language"].lower()

        language_counts[lang] = (
            language_counts.get(lang, 0) + 1
        )

    percentages = {}

    for lang, count in language_counts.items():
        percentages[lang] = (
            count / total_words * 100
            if total_words > 0
            else 0
        )

    return language_counts, percentages

In [19]:
all_results = []
segment_details = {}

for i, segment in enumerate(segments, start=1):

    segment_name = f"segment {i}"
    segment_text = segment["text"].strip()

    print(f"\nProcessing {segment_name}")
    print(segment_text)

    # ========================================================
    # CALL LLM
    # ========================================================

    prediction = detect_segment_languages(segment_text)

    predicted_words = prediction["words"]


    # ========================================================
    # HITUNG PERSENTASE DENGAN PYTHON
    # ========================================================

    counts, percentages = calculate_language_percentages(
        predicted_words
    )


    # simpan detail JSON per segment
    segment_details[segment_name] = {
        "text": segment_text,
        "words": predicted_words,
        "counts": counts,
        "percentages": percentages
    }


    # ========================================================
    # FORMAT UNTUK TABEL AKHIR
    # ========================================================

    for language, percentage in percentages.items():

        all_results.append({
            "segment": segment_name,
            "language": language,
            "percentage": f"{percentage:.1f}%"
        })


Processing segment 1
Fire is traffic. know what mean?

Processing segment 2
Yeah. I know. right. We're going Alandro.

Processing segment 3
Welcome back, Welcome back, little knee guards.

Processing segment 4
Oh my God. I don't know what compelled me to say that I'm sorry.

Processing segment 5
right, I'm good.


In [20]:
result_df = pd.DataFrame(all_results)

display(result_df)

,segment,language,percentage
0,segment 1,en,100.0%
1,segment 2,en,100.0%
2,segment 3,en,100.0%
3,segment 4,en,100.0%
4,segment 5,en,100.0%


In [21]:
print(
    json.dumps(
        segment_details,
        indent=2,
        ensure_ascii=False
    )
)

{
  "segment 1": {
    "text": "Fire is traffic. know what mean?",
    "words": [
      {
        "word": "Fire",
        "language": "en"
      },
      {
        "word": "is",
        "language": "en"
      },
      {
        "word": "traffic.",
        "language": "en"
      },
      {
        "word": "know",
        "language": "en"
      },
      {
        "word": "what",
        "language": "en"
      },
      {
        "word": "mean?",
        "language": "en"
      }
    ],
    "counts": {
      "en": 6
    },
    "percentages": {
      "en": 100.0
    }
  },
  "segment 2": {
    "text": "Yeah. I know. right. We're going Alandro.",
    "words": [
      {
        "word": "Yeah.",
        "language": "en"
      },
      {
        "word": "I",
        "language": "en"
      },
      {
        "word": "know.",
        "language": "en"
      },
      {
        "word": "right.",
        "language": "en"
      },
      {
        "word": "We're",
        "language": "en"
      },
     

In [22]:
for segment_name, data in segment_details.items():

    print("=" * 70)
    print(segment_name)
    print("Text:", data["text"])
    print()

    words_df = pd.DataFrame(data["words"])

    display(words_df)

    print("Counts:", data["counts"])
    print("Percentages:", data["percentages"])

segment 1
Text: Fire is traffic. know what mean?



,word,language
0,Fire,en
1,is,en
2,traffic.,en
3,know,en
4,what,en
5,mean?,en


Counts: {'en': 6}
Percentages: {'en': 100.0}
segment 2
Text: Yeah. I know. right. We're going Alandro.



,word,language
0,Yeah.,en
1,I,en
2,know.,en
3,right.,en
4,We're,en
5,going,en
6,Alandro.,en


Counts: {'en': 7}
Percentages: {'en': 100.0}
segment 3
Text: Welcome back, Welcome back, little knee guards.



...

Counts: {'en': 7}
Percentages: {'en': 100.0}
segment 4
Text: Oh my God. I don't know what compelled me to say that I'm sorry.



,word,language
0,Oh,en
1,my,en
2,God.,en
3,I,en
4,don't,en
5,know,en
6,what,en
7,compelled,en
8,me,en
9,to,en


Counts: {'en': 14}
Percentages: {'en': 100.0}
segment 5
Text: right, I'm good.



,word,language
0,right,en
1,",",en
2,I'm,en
3,good,en
4,.,en


Counts: {'en': 5}
Percentages: {'en': 100.0}


In [26]:
import pandas as pd

PARQUET_PATH = "/content/sessions_lang_transcript.parquet"

df = pd.read_parquet(PARQUET_PATH)

pd.reset_option("all")

display(df.head(20))

/tmp/ipykernel_2003/3727752863.py:7: FutureWarning: data_manager option is deprecated and will be removed in a future version. Only the BlockManager will be available.
  pd.reset_option("all")
/tmp/ipykernel_2003/3727752863.py:7: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  pd.reset_option("all")


,gamesession_id,user_id,game_name,model_type,created_at,lang_detected,lang_probability,transcript_segments
0,140597768,830382,Minecraft,gen10,2026-08-04 13:05:17,en,0.9990,"[{'text': ' Fire is traffic. know what mean?',..."
1,140602553,836604,Escape from Tarkov,gen10,2026-08-04 16:37:03,en,0.9980,"[{'text': ' You shoot the locks off.', 'timest..."
2,140595413,820850,Counter Strike2,gen10,2026-08-04 14:43:56,ru,0.9873,[{'text': ' кушать готовил очень вкусненько но...
3,140603946,693656,IRL,gen10,2026-08-04 17:27:53,de,0.8755,"[{'text': ' bin nicht geschissen, oder? Vollid..."
4,140603242,824533,Dead by Daylight,gen10,2026-08-04 17:01:12,en,0.9946,"[{'text': ' Welcome Phantom Ganja 420.', 'time..."
5,140605350,837970,Cuphead,gen10,2026-08-04 21:33:32,ru,0.9150,[{'text': ' запустив яу назарчик ты что пиши д...
6,140601039,837783,osu,gen10,2026-08-04 16:01:36,en,0.9839,"[{'text': ' Let's get active.', 'timestamp': [..."
7,140599775,837754,Avatar Legends: The Fighting Game,gen10,2026-08-04 15:05:56,pt,0.9961,[{'text': ' Os melhores preços em jogos e tecn...
8,140606661,838054,Grand Theft Auto V,gen5,2026-08-04 23:44:47,fr,0.9751,"[{'text': ' Oh là, là là, comment va les gens ..."
9,140605374,837974,The Mound: Omen of Cthulhu,gen5,2026-08-04 21:40:57,en,0.9971,"[{'text': ' Man, we're ringing at you the cove..."


In [27]:
import pandas as pd
import requests
import json
import re
from google.colab import userdata
from IPython.display import display

# ============================================================
# CONFIG
# ============================================================

PARQUET_PATH = "/content/sessions_lang_transcript.parquet"

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

MODEL = "tencent/hy-mt2-30b-a3b"
API_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

TARGET_SESSION_ID = 140595413


# ============================================================
# READ PARQUET
# ============================================================

df = pd.read_parquet(PARQUET_PATH)


# ============================================================
# FILTER SESSION
# ============================================================

session_df = df[
    df["gamesession_id"] == TARGET_SESSION_ID
]

if session_df.empty:
    raise ValueError(
        f"Session {TARGET_SESSION_ID} tidak ditemukan"
    )

session_row = session_df.iloc[0]

segments = session_row["transcript_segments"]


# ============================================================
# SESSION INFO
# ============================================================

print("Session ID          :", session_row["gamesession_id"])
print("Game                :", session_row["game_name"])
print("Internal lang       :", session_row["lang_detected"])
print("Internal probability:", session_row["lang_probability"])
print("Total segments      :", len(segments))

Session ID          : 140595413
Game                : Counter Strike2
Internal lang       : ru
Internal probability: 0.9873
Total segments      : 714


In [28]:
def detect_segment_languages(segment_text):

    prompt = f"""
Analyze every lexical word in the transcript segment below.

For every word, identify its language using ISO 639-1 codes where possible.

Examples:
- en = English
- ru = Russian
- id = Indonesian
- de = German
- fr = French
- es = Spanish
- pt = Portuguese
- uk = Ukrainian
- pl = Polish

Rules:
1. Classify every lexical word.
2. Preserve the original word order.
3. Do not add words.
4. Do not use any external language label.
5. Infer the language ONLY from the transcript text.
6. Ignore punctuation as separate tokens.
7. Return ONLY valid JSON.
8. Do NOT calculate percentages.

Transcript:
{segment_text}

Return exactly:

{{
  "words": [
    {{
      "word": "example",
      "language": "en"
    }}
  ]
}}
"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "reasoning": {
            "enabled": True
        }
    }

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    response_json = response.json()

    content = (
        response_json["choices"][0]["message"]["content"]
        .strip()
    )

    content = re.sub(r"^```json\s*", "", content)
    content = re.sub(r"^```\s*", "", content)
    content = re.sub(r"\s*```$", "", content)

    return json.loads(content)

In [29]:
def calculate_language_percentages(predicted_words):

    valid_words = [
        item
        for item in predicted_words
        if item.get("word", "").strip()
    ]

    total_words = len(valid_words)

    counts = {}

    for item in valid_words:

        language = item["language"].lower().strip()

        counts[language] = counts.get(language, 0) + 1

    percentages = {
        language: count / total_words * 100
        for language, count in counts.items()
    } if total_words else {}

    return counts, percentages

In [31]:
all_results = []
segment_details = {}

# BATASI HANYA 5 SEGMENT PERTAMA
segments_to_process = segments[:5]

for i, segment in enumerate(segments_to_process, start=1):

    segment_name = f"segment {i}"

    segment_text = str(
        segment["text"]
    ).strip()

    if not segment_text:
        continue

    print(
        f"[{i}/{len(segments_to_process)}] Processing {segment_name}"
    )

    # ========================================================
    # LLM HANYA MELIHAT TEXT
    # ========================================================

    prediction = detect_segment_languages(
        segment_text
    )

    predicted_words = prediction["words"]


    # ========================================================
    # PYTHON HITUNG PROPORSI
    # ========================================================

    counts, percentages = (
        calculate_language_percentages(
            predicted_words
        )
    )


    # ========================================================
    # SIMPAN DETAIL
    # ========================================================

    segment_details[segment_name] = {
        "text": segment_text,
        "words": predicted_words,
        "counts": counts,
        "percentages": percentages,
    }


    # ========================================================
    # FORMAT TABEL
    # ========================================================

    for language, percentage in percentages.items():

        all_results.append({
            "segment": segment_name,
            "text": segment_text,
            "language": language,
            "percentage": percentage,
        })

[1/5] Processing segment 1
[2/5] Processing segment 2
[3/5] Processing segment 3
[4/5] Processing segment 4
[5/5] Processing segment 5


In [32]:
result_df = pd.DataFrame(all_results)

result_df["percentage"] = (
    result_df["percentage"]
    .map(lambda x: f"{x:.1f}%")
)

display(result_df)

,segment,text,language,percentage
0,segment 1,кушать готовил очень вкусненько но приходите п...,ru,86.7%
1,segment 1,кушать готовил очень вкусненько но приходите п...,en,11.1%
2,segment 1,кушать готовил очень вкусненько но приходите п...,ja,2.2%
3,segment 2,"Вот он какой маленький сидит, сладенький, вкус...",ru,90.0%
4,segment 2,"Вот он какой маленький сидит, сладенький, вкус...",en,10.0%
5,segment 3,"Зря ты так с ним. Ну не знаю, на самом деле 50...",ru,86.7%
6,segment 3,"Зря ты так с ним. Ну не знаю, на самом деле 50...",en,13.3%
7,segment 4,"50 на 50, я бы сказал бы. А у тебя что, дела? ...",en,13.3%
8,segment 4,"50 на 50, я бы сказал бы. А у тебя что, дела? ...",ru,86.7%
9,segment 5,"О, нифига себе. А подожди, зачем тебе дорожка?...",ru,100.0%


In [33]:
for segment_name, data in segment_details.items():

    print("=" * 100)
    print(segment_name)
    print("Text:")
    print(data["text"])
    print()

    word_df = pd.DataFrame(
        data["words"]
    )

    display(word_df)

    print("Counts:")
    print(data["counts"])

    print("Percentages:")
    print(data["percentages"])

segment 1
Text:
кушать готовил очень вкусненько но приходите присаживайся реально надо подождем блин гта да ну на худа хики еще азиатик здорово привет моя киса я если что натурал сразу говорю я с азиатиком не заигрываю никогда я натурал уверенный в себе мужчина йоу я натурал уверенный



,word,language
0,кушать,ru
1,готовил,ru
2,очень,ru
3,вкусненько,ru
4,но,ru
5,приходите,ru
6,присаживайся,ru
7,реально,ru
8,надо,ru
9,подождем,ru


Counts:
{'ru': 39, 'en': 5, 'ja': 1}
Percentages:
{'ru': 86.66666666666667, 'en': 11.11111111111111, 'ja': 2.2222222222222223}
segment 2
Text:
Вот он какой маленький сидит, сладенький, вкусненький и все, Дэнс.



,word,language
0,Вот,ru
1,он,ru
2,какой,ru
3,маленький,ru
4,сидит,ru
5,сладенький,ru
6,вкусненький,ru
7,и,ru
8,все,ru
9,Дэнс,en


Counts:
{'ru': 9, 'en': 1}
Percentages:
{'ru': 90.0, 'en': 10.0}
segment 3
Text:
Зря ты так с ним. Ну не знаю, на самом деле 50 на 50, мужики.



...

Counts:
{'ru': 13, 'en': 2}
Percentages:
{'ru': 86.66666666666667, 'en': 13.333333333333334}
segment 4
Text:
50 на 50, я бы сказал бы. А у тебя что, дела? Нет, на дорожке?



,word,language
0,50,en
1,на,ru
2,50,en
3,я,ru
4,бы,ru
5,сказал,ru
6,бы,ru
7,А,ru
8,у,ru
9,тебя,ru


Counts:
{'en': 2, 'ru': 13}
Percentages:
{'en': 13.333333333333334, 'ru': 86.66666666666667}
segment 5
Text:
О, нифига себе. А подожди, зачем тебе дорожка? и так худая, я прям...



,word,language
0,О,ru
1,нифига,ru
2,себе,ru
3,А,ru
4,подожди,ru
5,зачем,ru
6,тебе,ru
7,дорожка,ru
8,и,ru
9,так,ru


Counts:
{'ru': 13}
Percentages:
{'ru': 100.0}


In [34]:
session_row["lang_detected"]
session_row["lang_probability"]

np.float64(0.9873)